# Publicar resultados dos experimentos no GitHub Pages

Execute ao final do experimento. A pasta de resultados deve já existir no Drive. Configure `GITHUB_TOKEN` nos **Segredos** do Google Colab. A primeira execução é uma **prévia**, sem envio. Ao revisar, altere `PUBLICAR = True`. Não publique credenciais nem os GeoTIFFs originais.

In [ ]:
# PUBLICADOR DE RESULTADOS CBERS-4A PARA GITHUB PAGES
# Uso: executar no Google Colab depois de gerar os resultados no Drive.
# Configure o secret GITHUB_TOKEN em Colab > ícone de chave (Secrets).
# NÃO coloque o token neste arquivo e NÃO envie tokens ao GitHub.
import os, json, base64, mimetypes
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import requests

# ===== 1) CONFIGURAÇÕES: EDITE A CADA NOVO EXPERIMENTO =====
REPOSITORIO = 'Carine-data-analyst/cbers4a-laboratorio'
ID_TESTE = 'experimento_003'  # novo ID para cada execução; sem espaços
TITULO = 'MobileNetV2 — experimento a catalogar'
DESCRICAO = 'Descrever objetivo e configuração do teste'
STATUS = 'A conferir'
TOTAL_AMOSTRAS = None         # preencher após confirmar no CSV
ACURACIA_TESTE = None         # fração de 0 a 1; preencher só após avaliar
CAMINHO_DRIVE = '/content/drive/MyDrive/CBERS4A/experimento_mobilenetv2_110_3_recortes'
PUBLICAR = False             # primeiro revisar a prévia; depois mudar para True
MAX_BYTES = 5 * 1024 * 1024 # limite conservador por arquivo
EXTENSOES = {'.png', '.jpg', '.jpeg', '.csv', '.json', '.md'}

# ===== 2) ENCONTRAR E REVISAR OS ARQUIVOS =====
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)
pasta = Path(CAMINHO_DRIVE)
if not pasta.is_dir():
    raise FileNotFoundError(f'Pasta não encontrada no Drive: {pasta}')
if not ID_TESTE.replace('_','').replace('-','').isalnum():
    raise ValueError('ID_TESTE deve ter apenas letras, números, _ ou -')

arquivos = sorted(p for p in pasta.iterdir() if p.is_file() and p.suffix.lower() in EXTENSOES and p.stat().st_size <= MAX_BYTES)
print('Arquivos elegíveis para publicação:')
for p in arquivos: print(f'  {p.name} ({p.stat().st_size:,} bytes)')
print('Arquivos NÃO elegíveis (grandes, GeoTIFF, modelos, subpastas):')
for p in pasta.iterdir():
    if p not in arquivos: print('  ',p.name)
if not arquivos: raise RuntimeError('Nenhum resultado publicável na pasta indicada.')

# ===== 3) CRIAR RESUMO SEM INVENTAR MÉTRICAS =====
hoje=datetime.now(ZoneInfo('America/Sao_Paulo')).date().isoformat()
base_destino=f'testes/{ID_TESTE}'
resumo={
  'id':ID_TESTE, 'titulo':TITULO, 'descricao':DESCRICAO, 'status':STATUS,
  'data':hoje, 'total_amostras':TOTAL_AMOSTRAS, 'acuracia_teste':ACURACIA_TESTE,
  'arquivos':[f'{base_destino}/{p.name}' for p in arquivos],
  'observacao':'Métricas e divisão dos dados devem ser conferidas no notebook original.'
}
print('\nResumo que será publicado:')
print(json.dumps(resumo,ensure_ascii=False,indent=2))
if not PUBLICAR:
    print('\nMODO PRÉVIA: nada foi enviado ao GitHub. Revise a lista e mude PUBLICAR=True para publicar.')
else:
    # ===== 4) AUTENTICAÇÃO RESTRITA VIA COLAB SECRETS =====
    token=userdata.get('GITHUB_TOKEN')
    if not token: raise RuntimeError('Configure o segredo GITHUB_TOKEN no Colab.')
    sess=requests.Session()
    sess.headers.update({'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
    api=f'https://api.github.com/repos/{REPOSITORIO}/contents'
    def obter(rel):
        r=sess.get(f'{api}/{rel}', params={'ref':'main'},timeout=30)
        if r.status_code==404:return None,None
        r.raise_for_status();obj=r.json()
        raw=base64.b64decode(obj['content']).decode('utf-8') if obj.get('encoding')=='base64' else None
        return obj.get('sha'),raw
    def enviar(rel,raw,binario=False):
        sha,_=obter(rel)
        payload={'message':f'Publica {ID_TESTE}: {rel}', 'branch':'main',
                 'content':base64.b64encode(raw if binario else raw.encode('utf-8')).decode('ascii')}
        if sha: payload['sha']=sha
        r=sess.put(f'{api}/{rel}',json=payload,timeout=45)
        if not r.ok: raise RuntimeError(f'Erro ao publicar {rel}: {r.status_code} {r.text[:250]}')
        print('Publicado:',rel)
    # Evita substituição involuntária de experimento anterior
    sha_antigo,_=obter(f'{base_destino}/resumo.json')
    if sha_antigo:raise RuntimeError(f'Já existe {ID_TESTE}. Escolha outro ID para preservar o histórico.')
    for p in arquivos:
        enviar(f'{base_destino}/{p.name}',p.read_bytes(),binario=True)
    enviar(f'{base_destino}/resumo.json',json.dumps(resumo,ensure_ascii=False,indent=2)+'\n')
    # Atualização do catálogo. Não sobrescrever outros experimentos.
    _,raw=obter('testes/catalogo.json')
    catalogo=json.loads(raw) if raw else {'versao':1,'experimentos':[]}
    if any(e.get('id')==ID_TESTE for e in catalogo['experimentos']):
        raise RuntimeError(f'{ID_TESTE} já está no catálogo. Revise antes de modificar.')
    catalogo['experimentos'].append({**{k:resumo[k] for k in ('id','titulo','descricao','status','data','total_amostras','acuracia_teste')},'pagina':f'{base_destino}/resumo.json'})
    enviar('testes/catalogo.json',json.dumps(catalogo,ensure_ascii=False,indent=2)+'\n')
    _,raw=obter('historico/atualizacoes.json')
    hist=json.loads(raw) if raw else {'versao':1,'atualizacoes':[]}
    hist['atualizacoes'].append({'data':hoje,'titulo':f'Teste publicado: {TITULO}','descricao':f'Resultados de {ID_TESTE} adicionados ao banco de testes.'})
    enviar('historico/atualizacoes.json',json.dumps(hist,ensure_ascii=False,indent=2)+'\n')
    print('\nCONCLUÍDO. GitHub Pages exibirá o novo teste após a publicação do commit.')
